In [ ]:
from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile 
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram

# Deutsch's algorithm
# Given an unknown function f : {0,1} -> {0,1}, work out whether it is constant
# or balanced.

# There are four possible functions:
#
# f0 is constant 0
# f1 is constant 1
# f2 is the identity function
# f3 is negation

# Deutsch's algorithm works with F, a quantum version of f, defined by
#
# F|x>|y> = |x>|f(x) XOR y>
#
# Explicitly define F0, F1, F2, F3 as circuits and convert them to gates.
# Ask the students to work out the circuits for F2 and F3, using F0 and F1 as templates.

# The lecture notes use x for the left qubit and y for the right qubit.
# Therefore x has index 1 and y has index 0.
# This is relevant when qBraid displays a state vector.

# f0 is constant 0
# F0 |0>|0> = |0>|0>
# F0 |0>|1> = |0>|1>
# F0 |1>|0> = |1>|0>
# F0 |1>|1> = |1>|1>

# F0 |x>|y> = |x>|y> so it's the identity function
# This means we don't need to add any gates to the circuit, just encapsulate it as a gate.

F0_circuit = QuantumCircuit(2)
F0 = circuit_to_gate(F0_circuit,label="F0")

# f1 is constant 1
# F1 |0>|0> = |0>|1>
# F1 |0>|1> = |0>|0>
# F1 |1>|0> = |1>|1>
# F1 |1>|1> = |1>|0>

# Therefore
# F1 |x>|y> = |x>|NOT(y)> so it's implemented by applying X(y)

F1_circuit = QuantumCircuit(2)
F1_circuit.x(0) # In qiskit, numbering of qubits starts from the right, so y is 0
F1 = circuit_to_gate(F1_circuit,label="F1")

# f2 is the identity function
# F2 |0>|0> = |0>|0>
# F2 |0>|1> = |0>|1>
# F2 |1>|0> = |1>|1>
# F2 |1>|1> = |1>|0>

# F2 |x>|y> = |x>|x XOR y> so it's implemented by applying CX(x,y)

F2_circuit = QuantumCircuit(2)
F2_circuit.cx(1,0) # Remember that numbering of qubits starts from the right
F2 = circuit_to_gate(F2_circuit,label="F2")

# f3 is negation
# F3 |0>|0> = |0>|1>
# F3 |0>|1> = |0>|0>
# F3 |1>|0> = |1>|0>
# F3 |1>|1> = |1>|1>

# F3 |x>|y> = |x>|NOT(x) XOR y> so it's implemented by applying X(x), CX(x,y), X(x)

F3_circuit = QuantumCircuit(2)
F3_circuit.x(1) # Remember that numbering of qubits starts from the right
F3_circuit.cx(1,0)
F3_circuit.x(1)
F3 = circuit_to_gate(F3_circuit,label="F3")

In [ ]:
# Choose which function we want to analyse

F = F2

In [ ]:
# Check that F is implemented correctly
# In this cell and the next three, make sure you understand why your
# definition of F is correct (or not).
# Test on |00>
test = QuantumCircuit(2) 
test.append(F,[0,1]) 
state = Statevector.from_int(0, 4) # 4 basis states on 2 qubits 
state = state.evolve(test) 
state.draw("latex")

In [ ]:
# Test on |01>
test = QuantumCircuit(2) 
test.x(0) # Remember that numbering of qubits starts from the right
test.append(F,[0,1]) 
state = Statevector.from_int(0, 4) # 4 basis states on 2 qubits 
state = state.evolve(test) 
state.draw("latex")

In [ ]:
# Test on |10>
test = QuantumCircuit(2) 
test.x(1) # Remember that numbering of qubits starts from the right
test.append(F,[0,1]) 
state = Statevector.from_int(0, 4) # 4 basis states on 2 qubits 
state = state.evolve(test) 
state.draw("latex")

In [ ]:
# Test on |11>
test = QuantumCircuit(2) 
test.x(0)
test.x(1) 
test.append(F,[0,1]) 
state = Statevector.from_int(0, 4) # 4 basis states on 2 qubits 
state = state.evolve(test) 
state.draw("latex")

In [ ]:
# The circuit for Deutsch's algorithm
# There are two qubits but we don't use them as inputs, just start them at |0>
# and manipulate them.
# We want to apply F to |+>|->
# So we prepare qubit 0 in state |-> and prepare qubit 1 in state |+>
# One classical bit is used to capture the measurement result.
# Ask the students to construct this circuit.

deutsch = QuantumCircuit(2,1)
deutsch.h(1)
deutsch.x(0)
deutsch.h(0) 
deutsch.append(F,[0,1]) 
deutsch.h(1) 
deutsch.measure([1],[0]) 
deutsch.draw("mpl")

In [ ]:
# Now simulate and plot the results.
# If f is constant (that is, F0 or F1) the result should always be 0.
# if f is balanced (that is, F2 or F3) the result should always be 1.

backend = BasicSimulator()
compiled = transpile(deutsch, backend)
job_sim = backend.run(compiled, shots=32)
result_sim = job_sim.result()
counts = result_sim.get_counts(compiled)
plot_histogram(counts)